# Run a trained policy in the MuJoCo simulator

This notebook runs a trained **ACT** policy on a simulated SO-101 arm with **Physical AI Runtime**. It shows the same steps as for a real robot, and it ends in the simulation's 3D viewer, where you
watch the policy work.

The arm sorts items in a **conveyor cell**: items ride a belt past it, and it has to drop each one into the right bin. **Red, blue and green items go into the bin of their colour, purple or cracked items into the reject bin.**

| Step | Physical AI Runtime |
|---|---|
| 1 | Start the simulation: `MuJoCoRobot` runs it in this notebook and serves its cameras and a 3D viewer |
| 2 | Load the policy: `InferenceModel.from_pretrained` |
| 3 | Connect: `RobotRuntime` with the robot, the cameras and the policy as action source |
| 4 | Run: `runtime.run()` / `runtime.stop()` |

The policy was trained with [Physical AI Studio](https://github.com/open-edge-platform/physical-ai-studio) on about 100 episodes recorded from the simulation's built-in demonstrator, then exported to OpenVINO.

## Setup

Until the packages are published, run this notebook from a checkout of the repository, with its own environment:

```bash
uv sync --all-packages --extra notebooks
uv run jupyter lab examples/tutorials
```

(Once `physicalai` and the plugin are on PyPI, `pip install "physicalai[plugin-mujoco,transport,notebooks]"` is all you need.)

In [ ]:
import os

# ACT policy, OpenVINO export. A Hub id, or a local folder with `manifest.json` via MODEL_DIR.
MODEL_REPO = os.environ.get("MODEL_DIR") or "iMaxxgx/act-so101-conveyor-sort"

## 1. Start the simulation

`MuJoCoRobot` runs the simulation inside this notebook. Unlike the real world, it moves only when it is driven: every observation the runtime reads advances the physics by one control tick (`rate_hz=30`, the policy's rate), so the belt and the items wait while the policy is stopped.

It also serves its cameras as MJPEG streams and a 3D viewer, each on a free port. `viewer_theme="compact"` starts the viewer's own control panel minimized.

In [3]:
import socket

from physicalai_mujoco_plugin import MuJoCoRobot


def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


http_port = free_port()
robot = MuJoCoRobot(
    profile="so101", scene="conveyor_sort", rate_hz=30,
    enable_viewer=True, viser_port=free_port(), viewer_theme="compact", http_port=http_port,
)
robot.connect()
robot.get_observation()  # one tick renders the first camera frames
http = f"http://127.0.0.1:{http_port}"

2026-10-20 00:11:08.914 | INFO     | physicalai_mujoco_plugin.robot:connect:272 - Loading MuJoCo scene /Users/xiangxi2/physicalai-wt-mujoco-notebook/packages/physicalai-mujoco-plugin/urdf/scenes/conveyor_sort/scene.xml with the so101 robot
2026-10-20 00:11:09.019 | DEBUG    | physicalai_mujoco_plugin.compose:attach_robot:348 - so101 options overridden by the scene: timestep=0.005
2026-10-20 00:11:09.037 | INFO     | physicalai_mujoco_plugin.conveyor:maybe_create:237 - Conveyor enabled (24 pool items, bins ['blue', 'green', 'red', 'reject'], belt 3.0 cm/s)
2026-10-20 00:11:09.215 | INFO     | physicalai_mujoco_plugin.robot:connect:292 - MuJoCo SO-101 connected (6 joints, timestep=0.002)


╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://127.0.0.1:8771   │
│   Websocket │ ws://127.0.0.1:8771     │
│             ╵                         │
╰───────────────────────────────────────╯

2026-10-20 00:11:09.883 | INFO     | physicalai_mujoco_plugin.viewer:_launch_viser:293 - 3D viewer: http://127.0.0.1:8771
2026-10-20 00:11:10.069 | INFO     | physicalai_mujoco_plugin.cameras:start:216 - Camera started: wrist (640x480@30 fps, own thread)
2026-10-20 00:11:10.070 | INFO     | physicalai_mujoco_plugin.cameras:start:216 - Camera started: overview (640x480@30 fps, own thread)
2026-10-20 00:11:10.171 | INFO     | physicalai_mujoco_plugin.control:_start_http_server:564 - HTTP camera server running at http://127.0.0.1:8770


## 2. Load the policy

`InferenceModel.from_pretrained` downloads the policy package from the Hugging Face Hub (an OpenVINO model plus a `manifest.json` that says which inputs it needs and how to preprocess them) and compiles it.

In [4]:
from physicalai.inference import InferenceModel

policy = InferenceModel.from_pretrained(MODEL_REPO, backend="openvino", device="CPU")

2026-10-20 00:11:10.265 | INFO     | physicalai.inference.model:__init__:109 - Loading policy from /Users/xiangxi2/pr353/hf-model
2026-10-20 00:11:10.266 | INFO     | physicalai.inference.model:__init__:136 - Loaded manifest: policy=act, backend=openvino, device=CPU
2026-10-20 00:11:10.266 | INFO     | physicalai.inference.model:__init__:146 - Compiling openvino model on CPU...
2026-10-20 00:11:10.397 | INFO     | physicalai.inference.adapters.openvino:_compile_config:51 - Using f32 inference precision on ARM CPU (set INFERENCE_PRECISION_HINT to override)
2026-10-20 00:11:10.838 | INFO     | physicalai.inference.model:__init__:148 - Model compiled in 0.6s
2026-10-20 00:11:10.863 | INFO     | physicalai.inference.model:__init__:169 - Loaded 1 preprocessors, 0 postprocessors, 0 callbacks
2026-10-20 00:11:10.863 | INFO     | physicalai.inference.model:__init__:181 - InferenceModel ready (chunk_size=1, total 0.6s)


## 3. Connect the robot, the cameras and the policy

A `RobotRuntime` ties them together, as on hardware:

- the **robot**: the `MuJoCoRobot` from step 1,
- the **cameras**: the simulation's MJPEG streams, served like a real robot's IP cameras. Their names (`overview`, `gripper`) must match the policy's inputs,
- the **action source**: `PolicySource` runs the policy. ACT predicts a chunk of 100 actions, and `request_threshold=0.95` asks for a fresh chunk after about every second action, which this policy needs (it does not recover from a missed grasp by itself),
- the **loop rate**: 30 Hz.

In [5]:
from physicalai.capture import IPCamera
from physicalai.runtime import ChunkedActionQueue, LerpSmoother, PolicySource, RobotRuntime, SyncExecution

runtime = RobotRuntime(
    robot=robot,
    cameras={
        "overview": IPCamera(url=f"{http}/cameras/overview/mjpeg", width=640, height=480, fps=30),
        "gripper": IPCamera(url=f"{http}/cameras/wrist/mjpeg", width=640, height=480, fps=30),
    },
    action_source=PolicySource(
        model=policy,
        execution=SyncExecution(request_threshold=0.95),
        action_queue=ChunkedActionQueue(smoother=LerpSmoother()),
    ),
    fps=30.0,
)
runtime.connect()  # connects the cameras; the robot already is

2026-10-20 00:11:14.329 | INFO     | physicalai.capture.cameras.ip._camera:_wait_first_frame:188 - IP camera http://127.0.0.1:8770/cameras/overview/mjpeg connected (640x480)
2026-10-20 00:11:14.332 | INFO     | physicalai.capture.cameras.ip._camera:_wait_first_frame:188 - IP camera http://127.0.0.1:8770/cameras/wrist/mjpeg connected (640x480)


## 4. Run it

The next cell shows three buttons, the simulation's 3D viewer (drag to orbit, scroll to zoom) and, beside it, the two cameras the policy sees:

- **Run policy** runs `runtime.run()` in a thread, **Stop** calls `runtime.stop()`. The simulation advances only while the policy runs, so the belt waits for it, and the viewer and the cameras stand still in between.
- **Reset** (while stopped) clears the belt and starts a new episode.

The score is in the viewer's control panel (**Conveyor**): items that ended up in the bin they belong in (**correct**), in a wrong bin (**wrong**), or off the belt (**missed**). Its **Belt speed** sets how fast items arrive: the policy was trained at 3 cm/s, so expect more misses when you speed it up. Changes made in that panel take effect once the policy runs.

> Running Jupyter on another machine? The viewer and camera URLs point at *that* machine's `localhost`. Forward the ports, or create the robot with `viser_host="0.0.0.0", http_host="0.0.0.0"`.

In [6]:
import threading

import ipywidgets as widgets
from IPython.display import HTML, display


def on_run(_):
    global run
    run = threading.Thread(target=runtime.run, daemon=True)
    run.start()
    run_button.disabled, stop_button.disabled, reset_button.disabled = True, False, True


def on_stop(_):
    runtime.stop()
    run.join()
    run_button.disabled, stop_button.disabled, reset_button.disabled = False, True, False


def on_reset(_):
    robot.reset()
    robot.get_observation()  # one tick shows the cleared scene


run_button = widgets.Button(description="Run policy", icon="play", button_style="success")
stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger", disabled=True)
reset_button = widgets.Button(description="Reset", icon="refresh")
run_button.on_click(on_run)
stop_button.on_click(on_stop)
reset_button.on_click(on_reset)

display(widgets.HBox([run_button, stop_button, reset_button]))
display(HTML(f"""<div style="display:flex;gap:8px">
  <iframe src="{http}/viewer" style="flex:1;height:540px;border:1px solid #ccc"></iframe>
  <div style="width:360px"><img src="{http}/cameras/overview/mjpeg" width="100%"><img src="{http}/cameras/wrist/mjpeg" width="100%"></div>
</div>"""))

Shut down: stop the policy if it runs, then disconnect the runtime, which also stops the simulation.

In [ ]:
if not stop_button.disabled:
    stop_button.click()  # stop the policy first
runtime.disconnect()

## Where next

- Start another robot or scene: `physicalai-mujoco profiles` lists the robots, and the [plugin README](../../packages/physicalai-mujoco-plugin/README.md) describes the scenes.
- Record your own demonstrations, train in Physical AI Studio and point `MODEL_REPO` at your own export.
- Run the same policy from the command line: `physicalai run --config runtime.yaml`.